In [1]:
import torch
import torch.nn as nn



In [2]:
from torch.utils.tensorboard import SummaryWriter
writer = SummaryWriter('runs/vae_chunk10_mean')

In [3]:
import numpy as np
from scipy.sparse import load_npz
datapath = '../../../data/CellCensus/3m_human_chunk_10.npz'
spare_matrix = load_npz(datapath)
dense_array = spare_matrix.toarray()

dense_tensor = torch.from_numpy(dense_array)

In [4]:
print(dense_tensor.size())

torch.Size([100000, 60664])


In [5]:
from torch.utils.data import Dataset

class MyDataset(Dataset):
    def __init__(self, tensor):
        self.tensor = tensor

    def __getitem__(self, index):
        x = self.tensor[index]
        return x

    def __len__(self):
        return self.tensor.size(0)
       

In [6]:
from torch.utils.data import random_split, DataLoader

dataset = MyDataset(dense_tensor)

train_len = int(0.8 * len(dataset))  # 80% for training
test_len = len(dataset) - train_len  # 20% for testing


train_data, test_data = random_split(dataset, [train_len, test_len])


train_loader = DataLoader(train_data, batch_size=256, shuffle=True)
test_loader = DataLoader(test_data, batch_size=256, shuffle=False)

In [7]:
import sys
sys.path.append('../../../')
from model import VAE


device = torch.device('cuda' if torch.cuda.is_available(
) else 'mps' if torch.backends.mps.is_available() else 'cpu')

print("Use device: ", device)

input_dim = dense_tensor.size(1) 
print("input_dim: ", input_dim)


model = VAE(
    encoder= nn.Sequential(
        nn.Linear(input_dim, 512),
        nn.ReLU(),
        nn.Linear(512, 256),
        nn.ReLU(),
    ),
    decoder= nn.Sequential(
        nn.Linear(64, 256),
        nn.ReLU(),
        nn.Linear(256, 512),
        nn.ReLU(),
        nn.Linear(512, input_dim),
        nn.Sigmoid()
    ),
    mean = nn.Linear(256, 64),
    var = nn.Linear(256, 64)
)

opt = torch.optim.Adam(model.parameters(), lr=1e-3)

model.to(device)


Use device:  mps
input_dim:  60664
VAE model initialized


VAE(
  (encoder): Sequential(
    (0): Linear(in_features=60664, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=256, bias=True)
    (3): ReLU()
  )
  (decoder): Sequential(
    (0): Linear(in_features=64, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=60664, bias=True)
    (5): Sigmoid()
  )
  (mean): Linear(in_features=256, out_features=64, bias=True)
  (var): Linear(in_features=256, out_features=64, bias=True)
)

In [8]:
def loss_function(x, x_hat, mean, log_var):
    reproduction_loss = nn.functional.mse_loss(x_hat, x)
    KLD = torch.mean(- 0.5 * torch.sum(1 + log_var - mean.pow(2) - log_var.exp()))
    return reproduction_loss, KLD

In [9]:

model.train()
for i in range(100):
    overall_loss = 0
    overall_rec_loss = 0
    overall_kld_loss = 0
    for x in train_loader:
        batch_size = x.size(0)
        x = x.to(device)
        opt.zero_grad()
        _, mean, log_var, _, x_hat = model(x)
        loss_rec, loss_kld = loss_function(x, x_hat, mean, log_var)
        loss = loss_rec + loss_kld
        loss.backward()
        opt.step()
        mean_rec = loss_rec.item()
        mean_kld = loss_kld.item()
        overall_loss += mean_rec + mean_kld
        overall_rec_loss += mean_rec
        overall_kld_loss += mean_kld
    writer.add_scalar('Loss/overall', overall_loss/len(train_loader), i)
    writer.add_scalar('Loss/rec', overall_rec_loss/len(train_loader), i)
    writer.add_scalar('Loss/kld', overall_kld_loss/len(train_loader), i)
    print(f"Epoch {i}, Loss: {overall_loss/len(train_loader)}, Rec: {overall_rec_loss/len(train_loader)}, KLD: {overall_kld_loss/len(train_loader)}")

model.eval()

with torch.no_grad():
    rec_loss_eval = 0
    kld_loss_eval = 0
    for x in test_loader:
        x = x.to(device)
        _, _, _, _, x_hat = model(x)
        loss_rec = nn.functional.mse_loss(x_hat, x, reduction='mean')
        rec_loss_eval += loss_rec.item()
    writer.add_scalar('Loss/rec_eval', rec_loss_eval/len(test_loader), i)
    print(f"Loss: {rec_loss_eval/len(test_loader)}")



Epoch 0, Loss: 64.68144932534463, Rec: 0.08677755827054429, KLD: 64.59467176707408
Epoch 1, Loss: 0.1980482803556485, Rec: 0.08187115592316697, KLD: 0.11617712443248152
Epoch 2, Loss: 0.109655662966422, Rec: 0.08178047643016322, KLD: 0.027875186536258784
Epoch 3, Loss: 0.08856329069541286, Rec: 0.08175190250142314, KLD: 0.006811388193989714
Epoch 4, Loss: 0.08288286892941203, Rec: 0.08173692421600841, KLD: 0.0011459447134036225
Epoch 5, Loss: 0.08188545511077387, Rec: 0.08172617055261477, KLD: 0.00015928455815909388
Epoch 6, Loss: 0.08187050453294961, Rec: 0.08171483253042539, KLD: 0.0001556720025242327
Epoch 7, Loss: 0.08186491774008296, Rec: 0.08171430318214642, KLD: 0.00015061455793654956
Epoch 8, Loss: 0.08190818890310324, Rec: 0.08169634466449292, KLD: 0.00021184423861031335
Epoch 9, Loss: 0.08194006165376487, Rec: 0.08168751334610838, KLD: 0.0002525483076564801
Epoch 10, Loss: 0.08199527571662166, Rec: 0.08180453675909165, KLD: 0.00019073895753001253
Epoch 11, Loss: 0.08191494862

In [10]:
# import os
# import time 
# date = time.strftime("%Y%m%d-%H%M%S")
# current = os.getcwd()
# os.makedirs(f"./trained_model/vae_{date}_mean", exist_ok=True)
# torch.save(model, f"./trained_model/vae_{date}_mean/vae_{date}.pt")
# torch.save(model.state_dict(),
#            f"./trained_model/vae_{date}_mean/vae_{date}_state_dict.pt")
# torch.save(opt.state_dict(),
#            f"./trained_model/vae_{date}_mean/optimizer_{date}_state_dict.pt")

# writer.close()